In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Any

In [2]:
@torch.no_grad()
def deepseek_dist(x, y):
    x, y = x.double(), y.double()
    denom = (x * x + y * y).sum()
    sim = 2 * (x * y).sum() / denom
    return (1 - sim).item()

In [3]:
from matmul_fp8_fp8 import alloc_and_quant, scaled_matmul_fp8_fp8


def matmul_autocast_fp8(a, b, transpose_right=False):
    a_fp8, a_scales = alloc_and_quant(a)
    b_fp8, b_scales = alloc_and_quant(b)
    if transpose_right:
        return scaled_matmul_fp8_fp8(a_fp8, b_fp8.T, a_scales, b_scales.T.contiguous())
    return scaled_matmul_fp8_fp8(a_fp8, b_fp8, a_scales, b_scales)

In [4]:
class Fp8LinearFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None):
        ctx.save_for_backward(input, weight, bias)
        
        output = matmul_autocast_fp8(input, weight, transpose_right=True)
        # output = input @ weight.T

        if bias is not None:
            output += bias
        
        return output
    
    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        input, weight, bias = ctx.saved_tensors
        
        grad_input = grad_weight = grad_bias = None
        
        if ctx.needs_input_grad[0]:
            grad_input = matmul_autocast_fp8(grad_output, weight)
            # grad_input = grad_output @ weight
        
        if ctx.needs_input_grad[1]:
            grad_weight = matmul_autocast_fp8(grad_output.T.contiguous(), input)
            # grad_weight = grad_output.T @ input
        
        if bias is not None and ctx.needs_input_grad[2]:
            grad_bias = grad_output.sum(dim=0)

        return grad_input, grad_weight, grad_bias

class Fp8Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        
        self.in_features = in_features
        self.out_features = out_features
        
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features)
        )
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.register_parameter('bias', None)


    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return Fp8LinearFunction.apply(input, self.weight, self.bias)
    

    def extra_repr(self) -> str:
        return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'



In [5]:
IN = 1024
OUT = 2048


batch_size = 512
torch.manual_seed(0)
x = torch.randn(batch_size, IN, requires_grad=True, device='cuda')
torch.manual_seed(0)
y = torch.randn(batch_size, IN, requires_grad=True, device='cuda')

target = torch.randn(batch_size, OUT, device='cuda')


model = torch.nn.Linear(IN, OUT, bias=False)
model_fast = Fp8Linear(IN, OUT, bias=False)
with torch.no_grad():
    model_fast.weight.copy_(model.weight.data.detach().clone())

model = model.cuda() 
model_fast = model_fast.cuda()

output = model(x)
loss = F.mse_loss(output, target)
loss.backward()

output_fast = model_fast(y)
loss_fast = F.mse_loss(output_fast, target)
loss_fast.backward()

In [6]:
deepseek_dist(output, output_fast)

0.0006301123337706382

In [7]:
deepseek_dist(x.grad, y.grad)

0.0006449856460623016

In [8]:
deepseek_dist(model.weight.grad, model_fast.weight.grad)

0.0007944505669280622